# Generador de imágenes — amused/amused-512 (Google Colab, GPU T4)
Entorno: *Entorno de ejecución → Cambiar tipo → T4 GPU*.
Celdas 1-3 generan la imagen del dinosaurio seleccionado (Doractodon). Celda 4 levanta un servidor HTTP con ngrok
para que la app web pida imágenes de nuevos dinosaurios.

Contrato del servidor: `POST /generate {"prompt": str, "seed": int|null}` → `image/png`.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors fastapi uvicorn pyngrok nest_asyncio
!git clone -q https://github.com/cofferr/LabDino 2>/dev/null; %cd LabDino

In [ ]:
import sys, json, torch
sys.path.insert(0, "image_generator")
from generate_image import load_pipeline, build_prompt, generate
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
pipe = load_pipeline()

In [ ]:
data = json.load(open("ollama/identity.json"))
prompt = build_prompt(data["identity"])
print(prompt)
img = generate(pipe, prompt, seed=0)
img.save(f"{data['name']}.png"); img

### Servidor para la app web
Poner el token de ngrok en *Secrets* de Colab con el nombre `NGROK_TOKEN` (nunca en el notebook).
La URL impresa va en `IMAGE_API_URL` del backend, con `IMAGE_NGROK_SKIP_WARNING=true`.

In [ ]:
import io, threading, nest_asyncio, uvicorn
from fastapi import FastAPI
from fastapi.responses import Response
from pydantic import BaseModel
from pyngrok import ngrok
from google.colab import userdata

api = FastAPI()
gpu = threading.Lock()

class Req(BaseModel):
    prompt: str
    seed: int | None = None

@api.get("/health")
def health():
    return {"ok": True, "model": "amused/amused-512"}

@api.post("/generate")
def gen(req: Req):
    with gpu:
        im = generate(pipe, req.prompt[:400], seed=req.seed)
    buf = io.BytesIO(); im.save(buf, format="PNG")
    return Response(buf.getvalue(), media_type="image/png")

ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))
print("IMAGE_API_URL =", ngrok.connect(8001).public_url)
nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(api, host="0.0.0.0", port=8001), daemon=True).start()